# SAMPO2 Live Inference & Backtest Notebook
This notebook allows running the full SAMPO2 pipeline interactively. You can fetch new data from IBKR, process features, and run the deployed PPO Agent.

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import tensorflow as tf
import matplotlib.pyplot as plt
import json
from pathlib import Path
from stable_baselines3 import PPO
import sys
import os

# Ensure project root is in path
project_root = Path.cwd().parent.parent if 'sampo2' in Path.cwd().parts else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from sampo2.config import OUTPUT_DIR, COMBINED_DATA_FILE
from sampo2.data.aggregator import SampoDataAggregator
from sampo2.data.garch import GarchVolatilityModel
from sampo2.models.pnn import ParallelNeuralNetwork
from sampo2.data.renko_extractor import RenkoIndicator, TradeBar
from sampo2.env.trading_env import TradingEnv
from sampo2.agents.risk_evaluator import RiskEvaluator

# Configuration
IB_CONFIG = {'host': '127.0.0.1', 'port': 7497, 'client_id': 1}
LOOKBACK_DAYS = 60
BACKTEST_YEAR = 2017 # Set to None for Live Mode, or Year (int) for backtest

## 1. Data Ingestion
Connect to IBKR TWS or fall back to mock file.

In [ ]:
df_raw = None

try:
    from sampo2.data.ib_aggregator import IBAggregator
    if BACKTEST_YEAR:
        print(f"Fetching history for {BACKTEST_YEAR}...")
        end_date = f"{BACKTEST_YEAR+1}0101 00:00:00"
        duration = "1 Y"
    else:
        print(f"Fetching Live history ({LOOKBACK_DAYS} days)...")
        end_date = ""
        duration = f"{LOOKBACK_DAYS} D"

    agg = IBAggregator(host=IB_CONFIG['host'], port=IB_CONFIG['port'], client_id=IB_CONFIG['client_id'])
    df_raw = agg.get_historical_data("EURUSD", sec_type="FOREX", bar_size="1 hour", 
                                     duration=duration, end_date=end_date)
    
    if not df_raw.empty:
        print(f"Fetched {len(df_raw)} rows from IBKR.")
    else:
        print("IBKR Fetch Empty/Failed.")
except Exception as e:
    print(f"IBKR Error: {e}")

if df_raw is None or df_raw.empty:
    print("Using Mock Data (Fallback)...")
    df_raw = pd.read_csv(OUTPUT_DIR / "history_2005_2009_ib.csv").tail(2000).reset_index(drop=True)
    if 'Time' in df_raw.columns:
        df_raw['Time'] = pd.to_datetime(df_raw['Time'], utc=True)

# Standardization
if 'Time' in df_raw.columns:
    df_raw['Time'] = pd.to_datetime(df_raw['Time'], utc=True)
if 'volume' not in df_raw.columns:
    if 'Volume' in df_raw.columns: df_raw.rename(columns={'Volume': 'volume'}, inplace=True)
    else: df_raw['volume'] = 0

print(f"Data Ready: {len(df_raw)} bars.")

## 2. Feature Engineering
Running Aggregation, GARCH, PNN, and Renko models.

In [ ]:
print("Processing Features...")
agg_tool = SampoDataAggregator(config="oanda.cfg")

# 1. Aggregation
df_h1 = agg_tool.run(instrument="EUR_USD", granularity="H1", raw_df=df_raw)

d_raw = df_raw.copy().set_index('Time').resample('1D').agg(
    {'Open':'first','High':'max','Low':'min','Close':'last','volume':'sum'}).dropna().reset_index()
if 'volume' not in d_raw.columns: d_raw['volume'] = 0
df_d = agg_tool.run(instrument="EUR_USD", granularity="D", raw_df=d_raw)

m_raw = df_raw.copy().set_index('Time').resample('15min').ffill().reset_index()
df_m15 = agg_tool.run(instrument="EUR_USD", granularity="M15", raw_df=m_raw)

# 2. GARCH
garch = GarchVolatilityModel()
if 'returns' not in df_h1.columns: 
    df_h1['returns'] = df_h1['Close'].pct_change().fillna(0)
df_h1['volatility_h1'] = garch.fit_predict(df_h1['returns'])

# 3. PNN Embeddings
model_path = OUTPUT_DIR / "pnn_model_full.keras"
df_d_aligned = df_d.reindex(df_h1.index, method='ffill')
df_m15_aligned = df_m15.reindex(df_h1.index, method='ffill')
numeric_cols = df_h1.select_dtypes(include=[np.number]).columns
common_cols = [c for c in numeric_cols if c in df_d_aligned.columns and c in df_m15_aligned.columns]

X_h1 = df_h1[common_cols].fillna(0).values.astype('float32')
X_d = df_d_aligned[common_cols].fillna(0).values.astype('float32')
X_m15 = df_m15_aligned[common_cols].fillna(0).values.astype('float32')

def z_norm(x): return (x - x.mean(axis=0)) / (x.std(axis=0) + 1e-8)
X_h1, X_d, X_m15 = z_norm(X_h1), z_norm(X_d), z_norm(X_m15)

pnn = tf.keras.models.load_model(model_path)
fusion = pnn.get_layer('enriched_features')
extractor = tf.keras.Model(inputs=pnn.inputs, outputs=fusion.output)
embeddings = extractor.predict([X_h1, X_d, X_m15], verbose=0)
emb_df = pd.DataFrame(embeddings, columns=[f'pnn_{i}' for i in range(embeddings.shape[1])])
emb_df.index = df_h1.index
final_df = df_h1.join(emb_df, how='inner')

# 4. Renko Prediction
renko_model = xgb.XGBClassifier()
renko_model.load_model(OUTPUT_DIR / "renko_predictor_xgboost.json")

renko = RenkoIndicator(name="Renko50", block_points=0.0005, instrument="EURUSD")
renko_data = []
for idx, row in final_df.iterrows():
    bar = TradeBar(row.name, row['Open'], row['High'], row['Low'], row['Close'])
    renko.Update(bar)
    if renko.IsReady:
        state = renko.get_ObjectDictionary()
        d = {'Time': row.name, 'Close': row['Close'], 'Renko_Value': state['Ren_Value'], 
             'Blue': int(state['Blue']), 'Red': int(state['Red']), 'Yellow': int(state['Yellow']), 
             'Flag': state['Flag']}
        for col in row.index: 
            if col not in d and col != 'Time': d[col] = row[col]
        renko_data.append(d)
        
renko_df = pd.DataFrame(renko_data)
if not renko_df.empty:
    bricks = renko_df[renko_df['Renko_Value'].shift() != renko_df['Renko_Value']].copy()
    # Filter features
    drop_cols = ['Time', 'Class', 'Target_State', 'Target_Next_Blue'] + [c for c in bricks.columns if 'renko_' in c]
    X_renko = bricks.select_dtypes(include=[np.number]).drop(columns=[c for c in drop_cols if c in bricks.columns], errors='ignore')
    # Feature Match
    expected = renko_model.get_booster().feature_names
    if expected:
        missing = [f for f in expected if f not in X_renko.columns]
        if missing: 
            for m in missing: X_renko[m] = 0
        X_renko = X_renko[expected]

    probs = renko_model.predict_proba(X_renko)
    pred_df = pd.DataFrame(index=bricks.index)
    pred_df['Time'] = bricks['Time']
    pred_df['renko_prob_red'] = probs[:, 0]
    pred_df['renko_prob_blue'] = probs[:, 1]
    pred_df['renko_prob_ry'] = probs[:, 2]
    pred_df['renko_prob_by'] = probs[:, 3]
    
    # Merge
    final_df = final_df.sort_index()
    final_df['Time_Link'] = final_df.index
    pred_df = pred_df.sort_values('Time')
    df_merged = pd.merge_asof(final_df, pred_df, left_on='Time_Link', right_on='Time', direction='backward')
    df_merged.set_index('Time_Link', inplace=True)
    df_merged.fillna(0, inplace=True)
    print(f"Features Ready: {df_merged.shape}")
else:
    print("No Renko Bricks formed (Window too small?)")
    df_merged = final_df

## 3. Agent Execution
Backtesting the PPO Agent on the prepared data.

In [ ]:
# Load Model
ppo_path = OUTPUT_DIR / "ppo_final_model.zip"
model = PPO.load(ppo_path)

# Setup Env
df_numeric = df_merged.select_dtypes(include=[np.number, bool])
for col in df_numeric.select_dtypes(include=['bool']).columns:
    df_numeric[col] = df_numeric[col].astype(int)

# Load Params
try:
    with open(OUTPUT_DIR / "best_hyperparameters.json", 'r') as f:
        params = json.load(f)
        env_keys = ['vol_target', 'dd_limit', 'streak_kill', 'lambda_tc', 'lambda_vol', 'lambda_dd', 'lambda_streak']
        env_kwargs = {k: params[k] for k in env_keys if k in params}
except:
    env_kwargs = {}

env = TradingEnv(df_numeric, **env_kwargs)
obs, _ = env.reset()
done = False
equity_curve = [env.equity]

print("Running Simulation...")
while not done:
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, done, truncated, info = env.step(action)
    equity_curve.append(info['equity'])

print(f"Final Equity: {equity_curve[-1]:.2f}")

## 4. Evaluation & Visualization
Risk Metrics and Equity Curve.

In [ ]:
evaluator = RiskEvaluator(equity_curve)
print(evaluator.get_report())

plt.figure(figsize=(12, 6))
plt.plot(equity_curve)
plt.title(f"Equity Curve ({BACKTEST_YEAR if BACKTEST_YEAR else 'Live Inference'})")
plt.xlabel("Steps")
plt.ylabel("Balance")
plt.grid(True)
plt.show()